# OPF PII Benchmarks — independent review

<!-- VOICE: 1–2 sentence hook in your voice. Suggested punchline:
"OpenAI shipped a PII filter. NVIDIA published numbers for their PII detector on three public benchmarks. Nobody compared them fairly. So I did." -->

This notebook is the technical artifact behind the post. It reproduces an independent review of [OpenAI's Privacy Filter (OPF)](https://github.com/openai/privacy-filter) against the three benchmarks NVIDIA published GLiNER-PII numbers on — Argilla PII, AI4Privacy, and Nemotron-PII — in three modes each (penalty / fair span / fair typed).

## What this is, in plain English

<!-- VOICE: optional one-line zinger before the explainer. e.g. "Every assistant you talk to reads your address. Some companies are trying to do something about it." -->

Every AI assistant — ChatGPT, Claude, Copilot, whatever you're using — sees the full content of every message you send it. Names, emails, addresses, credit card numbers, API keys. A lot of that ends up in server logs, training pipelines, debugging tools, and analytics platforms by default. Companies are starting to ship **PII filters**: small models whose job is to spot personal info in text before it reaches anything that doesn't need it, so it can be redacted or audited.

This notebook benchmarks one of those filters: **OpenAI's Privacy Filter** (OPF; [model card][opf-card] · [code][opf-repo] · [weights][opf-hf]), released open-source in 2025. NVIDIA released a competing filter ([GLiNER-PII][gliner-hf]; [model card][gliner-card]) and published F1 scores against three public test sets. They never reported numbers for OPF and nobody else did either. So I ran OPF — as-shipped, no fine-tuning — against the same three test sets, using a hand-mapped label translation so the comparison is fair across slightly different category taxonomies.

A few terms that show up in the headline chart legend below:

- **F1 score** — runs 0 to 1. Higher is better. Above 0.7 is "actually useful in practice."
- **Three benchmarks** — Argilla PII, AI4Privacy, Nemotron-PII. No single test set captures everything that looks like PII, so we score against all three. (More on each in the [Label mapping decisions](#Label-mapping-decisions) section further down.)
- **Three scoring modes per benchmark** — there's no one right way to score this; each mode asks a different question:
  - *Penalty view (untyped × full)* — "across **all** PII in this dataset, what fraction does the filter spot at all?" Counts every gold span, including categories the filter wasn't trained on (job titles, age, IP addresses for OPF). Label-agnostic: the filter gets credit for any prediction that overlaps a gold span, regardless of category.
  - *Fair span (untyped × OPF-scope)* — drops out-of-scope categories from the gold set before scoring. "Is the filter good at catching PII in its design scope?" Still label-agnostic.
  - *Fair typed (typed × OPF-scope)* — additionally requires the filter to predict the **right** PII category, not just spot some PII. The strictest of the three by construction, and the closest match to the strict F1 NVIDIA published. This is the apples-to-apples comparison.

These three modes **aren't** ordered "strict to loose." Fair typed is always the strictest (it's strictly stricter than fair span — same gold set, plus a category-match requirement). But *penalty view vs fair span* depends on the dataset: stripping out-of-scope gold spans removes both filter misses (a win for the filter) and filter predictions that were silently overlapping out-of-scope golds (a loss). Which effect dominates is dataset-specific. Spoiler from the chart below: on AI4Privacy, **penalty view scores *higher* than fair span**. That's a real result, not a bug — explained in the "How to read this chart" caption right after the chart.

[opf-card]: https://cdn.openai.com/pdf/c66281ed-b638-456a-8ce1-97e9f5264a90/OpenAI-Privacy-Filter-Model-Card.pdf
[opf-repo]: https://github.com/openai/privacy-filter
[opf-hf]: https://huggingface.co/openai/privacy-filter
[gliner-card]: https://build.nvidia.com/nvidia/gliner-pii/modelcard
[gliner-hf]: https://huggingface.co/nvidia/gliner-PII

## TL;DR

<!-- VOICE: one sentence with the punchline + maybe the headline number. e.g. "On the strict-fair view, OPF lands at <X> vs NVIDIA's reported <Y> — but the per-category picture is much more interesting." -->

![Headline F1 by benchmark](vast_results/figs/headline_f1.png)

In [ ]:
# Headline chart. Reads from `vast_results/` (committed full-run metrics).
# The markdown image link above (`vast_results/figs/headline_f1.png`) is the
# static version used by GitHub's notebook viewer; this cell is the live render
# that always matches whatever's in `vast_results/`.

from opf_benchmarks.charts import load_metrics, plot_headline_f1

payloads = load_metrics("vast_results")
fig = plot_headline_f1(payloads)

**How to read this chart.** Each benchmark has 4 bars. The green bar is what NVIDIA reported for their PII model on that test set. The three blue bars are OPF's score under the three scoring modes from above. The rightmost ("fair typed") is the strictest by construction and the apples-to-apples comparison to NVIDIA's number; penalty view and fair span are different framings of the same predictions, not strictly stricter than each other.

**Wait — why does penalty view sometimes do better than fair span?** Look at AI4Privacy: penalty 0.85 vs fair span 0.82. Going from penalty → fair strips out-of-scope gold spans (`USERNAME`, `TIME`, `IP`, etc.) from the gold set. That removes filter misses (good for the filter) *but also* removes some gold spans the filter was predicting on top of (the untyped, label-agnostic scoring gave the filter credit for any overlap — once the gold is gone, those overlapping predictions become false positives). On AI4Privacy, the out-of-scope labels are mostly PII-adjacent (usernames sit next to emails, timestamps next to dates), so OPF predictions land near them; stripping the gold loses more precision than the recall improvement makes up for. On Argilla and Nemotron the opposite happens — many out-of-scope golds (age, gender, occupation) sit far from OPF's predictions, so dropping them mostly just removes misses and fair span F1 ticks up.

The story across the three benchmarks:

- **Argilla PII** — OPF lands a few points below NVIDIA across all three modes. Close, but NVIDIA wins. Worth knowing: Argilla's labels are mDeBERTa output, not human-validated — both filters are being scored against noisy ground truth.
- **AI4Privacy** — OPF *beats* NVIDIA. By a lot: ~16 points on the strict comparison. This is the result that broke my expectations going in.
- **Nemotron-PII** — OPF lags badly, but NVIDIA trained GLiNER-PII on this dataset's train split, so it's their home turf. The comparison isn't really fair — but it's the number they published, so it's the one we compare against.

The per-category and per-label charts below decompose where those numbers come from.

## Drill-down — where the headline comes from

Four views: per-OPF-category F1 (which kinds of PII OPF nails vs misses), per-source-label recall (which gold labels OPF catches at all, in-scope or not), token vs span F1 (how much the strict boundary policy costs), and precision-recall scatter (where each setup lands in P-R space).

### Per-OPF-category F1 (typed × OPF-scope)

The strict-fair view, split by which OPF category each span belongs to. OPF is essentially a perfect email detector across all three benchmarks (~0.97 F1), strong on `private_phone` and `account_number` (especially on AI4Privacy), and weakest on `private_url` (0.00–0.25 F1) and `private_address` (0.40–0.79). The AI4Privacy column being uniformly darker is what drives its headline F1 beat over NVIDIA.

In [ ]:
from opf_benchmarks.charts import plot_per_category_heatmap

plot_per_category_heatmap(payloads)

### Per-source-label recall — what OPF catches and what it misses

One panel per benchmark. Each bar is one of that benchmark's source labels; bar length is the fraction of that label's character span OPF predicted as *some* PII (label-agnostic). Blue = the label maps to an OPF category (OPF was trained for it). Grey = out of OPF's design scope (age, gender, vehicle VINs, user agents, etc.).

The visual story:

- **Argilla**: blue bars cluster near 1.0 (OPF doing its job); grey bars at the bottom near 0 (age, gender, occupation — OPF wasn't trained for them) are exactly the spans the penalty-view F1 docks OPF for missing.
- **AI4Privacy**: nearly every label is in OPF's scope, so OPF can mostly score points across the board.
- **Nemotron**: a mix, with technical identifiers (`vehicle_identifier`, `device_identifier`) appearing blue because they map to `account_number`; the long tail of demographics (`race_ethnicity`, `religious_belief`, `occupation`) is grey at zero recall.

In [ ]:
from opf_benchmarks.charts import plot_per_source_label_recall

plot_per_source_label_recall(payloads)

### Token vs span F1 — boundary-strictness penalty

The strict-span view requires OPF to predict the *exact* character span the gold annotation marks. The token-level (lenient) view gives partial credit for any token overlap with a gold span. The gap = how often OPF identifies the right PII but disagrees on the boundary.

Argilla shows the biggest drop (token 0.79 → span 0.67 on untyped × full): the noisy mDeBERTa-suggested labels there don't always agree with OPF's tokenization. AI4Privacy and Nemotron's gold annotations align much more cleanly with OPF's boundaries — the strict view costs almost nothing on the scope modes.

In [ ]:
from opf_benchmarks.charts import plot_token_vs_span

plot_token_vs_span(payloads)

### Precision vs Recall scatter

Where each (benchmark × mode) lands in precision-recall space; dashed lines are F1 isolines. AI4Privacy clusters in the top-right (high precision, high recall) — OPF makes few false alarms and catches most spans. Argilla's typed mode (square marker) drops precision noticeably — that's the typed-vs-untyped category-confusion penalty showing up: OPF finds the right span but assigns the wrong OPF category.

In [ ]:
from opf_benchmarks.charts import plot_pr_scatter

plot_pr_scatter(payloads)

## How this notebook works

The headline F1 chart above is **the real result**, computed against the full splits on an RTX 3090 and committed to `results/`. The cells below run the same pipeline **on a 100-example sample per benchmark** so a free-tier Colab session can finish in ~5 minutes. The sample run lands in `data_sample/` and `results_sample/`; it is for showing the pipeline works, not for the published numbers.

**Before running:**

1. Switch to a GPU runtime via *Runtime → Change runtime type → T4 GPU*. CPU works but takes longer.
2. Add a HuggingFace token as a Colab secret named `HF_TOKEN` (key icon in the left sidebar). Two of the three datasets (`ai4privacy/pii-masking-300k` and `nvidia/Nemotron-PII`) are gated — accept their terms on HuggingFace first, then create a read token at https://huggingface.co/settings/tokens.

Then *Runtime → Run all*.

In [ ]:
# --- Colab path: uncomment to clone the repo when running on Colab. ---
# Locally, skip this cell and use the next cell (which just verifies your cwd).

# import os
# REPO_DIR = "opf-benchmarks-geo"
# if not os.path.isdir(REPO_DIR):
#     !git clone https://github.com/CupOfGeo/opf-benchmarks-geo.git
# %cd {REPO_DIR}


In [ ]:
# --- Local sanity check ---
# The notebook lives at the repo root, so cwd should already be correct in both
# Colab (after the clone+%cd above) and locally. If this assertion fails, you
# opened the notebook from somewhere unexpected.

import os
from pathlib import Path

assert Path("scripts").is_dir() and Path("opf_benchmarks").is_dir(), (
    f"cwd looks wrong: {os.getcwd()} — open this notebook from the repo root."
)
print(f"cwd: {os.getcwd()}")

In [ ]:
# Install the package. On Colab this brings in the repo as `opf_benchmarks` +
# the OPF dependency. Locally, if you've already run `uv sync` (or `pip install -e .`),
# this is redundant and creates a stray `build/` directory — so we skip it.

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q .
else:
    print("Local mode — skipping `pip install .` (assumes the package is already installed).")

In [ ]:
import os

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except (ImportError, Exception):
    token = os.environ.get("HF_TOKEN")

if not token:
    raise RuntimeError(
        "HF_TOKEN not found. Add it as a Colab secret (key icon in the left sidebar, "
        "name it HF_TOKEN, toggle 'Notebook access' on). Two of the three datasets "
        "are gated — accept terms at https://huggingface.co/datasets/ai4privacy/pii-masking-300k "
        "and https://huggingface.co/datasets/nvidia/Nemotron-PII first, then create a "
        "read token at https://huggingface.co/settings/tokens."
    )

os.environ["HF_TOKEN"] = token
print("HF_TOKEN loaded.")

In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if DEVICE == "cpu":
    print("WARNING: no GPU detected — runs will be slow. Switch to T4 via Runtime → Change runtime type.")

## Knobs

This notebook runs on a **100-example sample per benchmark** so that a Colab session can finish in a few minutes. The full-size F1 numbers reported above (in the headline chart) come from a separate run against the full datasets on a GPU — the metrics for that run are committed to `results/` in the repo. The cells below write to `data/` and `results/` locally; when you clone fresh, they're empty, and your sample run populates them.

In [ ]:
MAX_EXAMPLES = 100      # raise this to re-run a bigger sample; the published numbers used the full splits
BENCHMARKS = ["argilla", "ai4privacy", "nemotron"]

## Label mapping decisions

OPF was trained on **8 categories** (listed below). Each benchmark uses its own taxonomy — Argilla has ~55 labels, AI4Privacy ~60, Nemotron ~45 — and they overlap only partially with what OPF actually targets. So we hand-mapped each benchmark's labels to one of OPF's 8, or marked them out-of-scope.

The key decisions:

- **All financial / government identifiers collapse into `account_number`.** OPF doesn't distinguish SSN from credit card from IBAN from tax ID, so neither do we. Everything from `BITCOINADDRESS` to `medical_record_number` lands here.
- **Every flavor of name collapses into `private_person`** — first/last/middle/given/surname/title/prefix all map to the single category.
- **Demographic attributes are out of scope.** Age, gender, race, religion, political views, sexuality, education, occupation — none are PII in OPF's framing. Counting them against OPF would mis-frame the comparison.
- **Technical identifiers are out of scope.** IP addresses, MACs, user agents, device IDs, vehicle VINs, license plates — OPF treats these as network/asset metadata, not personal info.
- **Job titles, currency amounts, GPS coordinates, biometrics are out of scope** for the same reason — potentially identifying-in-aggregate, but not what OPF was trained to detect.

Out-of-scope spans are **deliberately kept in `_full.jsonl` with their original label** — that's what powers the "untyped × full" (penalty view) F1 in the report, and the per-source-label recall breakdown that shows exactly which categories OPF doesn't cover. The `_opfscope.jsonl` files drop them so typed evaluation is well-defined.

Full mapping below — see [`opf_benchmarks/label_map.py`](https://github.com/CupOfGeo/opf-benchmarks-geo/blob/main/opf_benchmarks/label_map.py) for the source.

In [ ]:
from IPython.display import Markdown
from opf_benchmarks.label_map import MAPS, OPF_CATEGORIES

lines = ["### OPF's 8 target categories\n"]
for cat in sorted(OPF_CATEGORIES):
    lines.append(f"- `{cat}`")

lines.append("\n### Source labels grouped by OPF category\n")
for cat in sorted(OPF_CATEGORIES):
    lines.append(f"**`{cat}`**")
    for bench, table in MAPS.items():
        srcs = sorted(src for src, dst in table.items() if dst == cat)
        if srcs:
            lines.append(f"- *{bench}*: " + ", ".join(f"`{s}`" for s in srcs))
    lines.append("")

lines.append("### Out of OPF's scope\n")
lines.append(
    "These labels are kept in `_full.jsonl` with their original label (so the "
    "penalty-view F1 still counts them against OPF) and dropped from "
    "`_opfscope.jsonl` (so typed evaluation is well-defined):\n"
)
for bench, table in MAPS.items():
    oos = sorted(src for src, dst in table.items() if dst is None)
    lines.append(f"- *{bench}* ({len(oos)} labels): " + ", ".join(f"`{s}`" for s in oos))

Markdown("\n".join(lines))

In [ ]:
max_arg = f"--max-examples {MAX_EXAMPLES}" if MAX_EXAMPLES is not None else ""
!python -m scripts.download_datasets {max_arg} --benchmarks {" ".join(BENCHMARKS)}

In [ ]:
!python -m opf_benchmarks.run_eval --device {DEVICE} --benchmarks {" ".join(BENCHMARKS)}

In [ ]:
!python -m opf_benchmarks.aggregate

## Sample results

What the pipeline produces on the 100-example sample you just ran. The headline F1 numbers at the top of the notebook come from the full-size run in `results/`, not these.

In [ ]:
from IPython.display import Markdown

Markdown(open("results/REPORT.md").read())

## Bonus: run OPF on your own AI chats

<!-- VOICE: short intro to the personal-audit section. The post angle: "the benchmark is interesting; what catches your eye is what OPF flags in your own Claude Code / ChatGPT history." -->

A companion script reads your local Claude Code transcripts and runs OPF over them, then prints a category-by-category count of what it flagged. The script is in [`scripts/audit_claude_chats.py`](https://github.com/CupOfGeo/opf-benchmarks-geo/blob/main/scripts/audit_claude_chats.py) (added in a later phase — placeholder for now). Nothing personal leaves your machine; the script's default output is counts only.

## Sources & further reading

**OpenAI Privacy Filter (OPF)**

- [Model card (PDF)](https://cdn.openai.com/pdf/c66281ed-b638-456a-8ce1-97e9f5264a90/OpenAI-Privacy-Filter-Model-Card.pdf) — full technical details, training data, intended use, and limitations as stated by OpenAI
- [Source code (GitHub)](https://github.com/openai/privacy-filter) — the `opf` CLI and Python API; this notebook pins the commit in `pyproject.toml`
- [Model weights on HuggingFace](https://huggingface.co/openai/privacy-filter) — the 2.6 GB checkpoint OPF downloads on first eval

**NVIDIA GLiNER-PII**

- [Model card on build.nvidia.com](https://build.nvidia.com/nvidia/gliner-pii/modelcard) — the published strict-F1 numbers this notebook compares against come from here
- [Model weights on HuggingFace](https://huggingface.co/nvidia/gliner-PII)

**Benchmark datasets**

- [Argilla PII (`argilla/textcat-tokencat-pii-per-domain`)](https://huggingface.co/datasets/argilla/textcat-tokencat-pii-per-domain) — train split; labels are mDeBERTa output, not human-validated. ([Argilla org page](https://huggingface.co/argilla))
- [AI4Privacy (`ai4privacy/pii-masking-300k`)](https://huggingface.co/datasets/ai4privacy/pii-masking-300k) — gated; accept the dataset terms before running. ([AI4Privacy org page](https://huggingface.co/ai4privacy))
- [NVIDIA Nemotron-PII (`nvidia/Nemotron-PII`)](https://huggingface.co/datasets/nvidia/Nemotron-PII) — test split; gated; GLiNER-PII was trained on its train split, so OPF here is fully out-of-distribution while GLiNER-PII is in-distribution.

## (Optional) Download the sample report

Run the cell below in Colab to download the sample `REPORT.md` to your machine. Skipped automatically when not running in Colab.

In [ ]:
try:
    from google.colab import files
    files.download("results/REPORT.md")
except ImportError:
    print("Not running in Colab — skipping download.")